# T05 · Entropy sweep on the crate stage

We train one FC128 (Kartal) agent on the first stage of the 9×9 curriculum (collect coins, no bombs). Its
weights are then the starting point of $21$ short runs of the second stage (bomb crates to find coins). The runs
differ only in the entropy coefficient $c_H$, which stays constant within a run and ranges from $10^{-1}$ down to
$5\cdot10^{-4}$ on a log scale. Seed, budget and rewards are the same for all runs, so the differences show how
$c_H$ alone decides whether the agent learns to bomb, stays passive or keeps killing itself.

**Writes** `runs/a2c_entropy_sweep/`: `history.csv` (all runs, one row per iteration), `config.json` and
`collect.eqx`. **Used by** A08.

In [ ]:
import os

os.environ.setdefault("JAX_COMPILATION_CACHE_DIR", os.path.expanduser("~/.cache/jax"))
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.90")

import json
import pathlib
import time

import equinox as eqx
import jax
import numpy as np
import pandas as pd

from bomberman.agents import A2CObjective, PolicyAgent
from bomberman.agents.networks import KartalCNNActorCriticNetwork
from bomberman.game import ACTION
from bomberman.training import (
    CoinPickupReward,
    DeathPenalty,
    PrintLogger,
    StepPenalty,
    Training,
    UrgentCrateReward,
    UrgentStepPenalty,
    coin_task_settings,
    crate_task_settings,
)

HERE = pathlib.Path.cwd().resolve()  # run from the notebook's folder or from the repository root
EXP = next(p / "experiments" for p in (HERE, *HERE.parents) if (p / "experiments" / "README.md").exists())
OUT = EXP / "runs" / "a2c_entropy_sweep"
OUT.mkdir(parents=True, exist_ok=True)
print(f"backend: {jax.default_backend()} | output: {OUT}")

## Configuration

Boards and rewards are the same as in the 9×9 search (T02). The first stage uses the best learning rate and
entropy schedule that search found for FC128; the sweep uses its learning rate for the crate stage.

In [ ]:
SIZE = 9
SEED = 0
ENTROPY_COEFS = np.geomspace(1e-1, 5e-4, 21)  # c_H of each crate-stage run, constant during the run
COLLECT = dict(iterations=1000, learning_rate=1.6e-4, entropy=(8.7e-3, 5e-4))
SWEEP = dict(iterations=300, learning_rate=2.5e-3)
NUM_ENVS = 150

COLLECT_SETTINGS = coin_task_settings(size=SIZE, coins=8, max_steps=96)
COLLECT_REWARD = CoinPickupReward() + StepPenalty(step=0.02)
CRATE_SETTINGS = crate_task_settings(size=SIZE, coins=6, crate_density=0.18, max_steps=140)
CRATE_REWARD = CoinPickupReward() + UrgentCrateReward() + DeathPenalty(death=2.0) + UrgentStepPenalty()
ENTROPY_COEFS

## Agent and training

`TrackedA2C` is A2C that also logs the number of bombs planted per game.

In [ ]:
class TrackedA2C(A2CObjective):
    def __call__(self, network, batch):
        loss, metrics = super().__call__(network, batch)
        bombs = ((batch.action == ACTION.BOMB) & batch.mask).sum(-1).mean()
        return loss, {**metrics, "bombs": bombs}


def train(agent, key, settings, reward, *, iterations, learning_rate, entropy, forbid_bomb=False):
    """Train one stage. c_H decays linearly from entropy[0] to entropy[1]; equal values keep it constant."""
    c0, c1 = entropy
    agent = agent.configure(objective=agent.objective.configure(entropy_coef=float(c0), entropy_final=float(c1)))
    trainer = Training(settings, [agent], reward_fn=reward, forbid_bomb=forbid_bomb, num_envs=NUM_ENVS,
                       learning_rate=learning_rate,
                       logger=PrintLogger(every=50, keys=("agent0/coins", "agent0/death", "agent0/bombs")))
    agent, *_ = trainer.fit(key, iterations)
    history = pd.DataFrame({k.replace("agent0/", ""): np.asarray(v) for k, v in trainer.history.items()})
    return agent, history.assign(iteration=np.arange(len(history)))

## Stage 1, once

In [ ]:
init_key, collect_key, sweep_key = jax.random.split(jax.random.key(SEED), 3)
agent = PolicyAgent(KartalCNNActorCriticNetwork.for_settings(CRATE_SETTINGS, init_key),
                    TrackedA2C(gamma=0.99, entropy_coef=0.01, entropy_final=0.01, value_coef=0.5, n_steps=5))
start = time.time()
agent, collect_history = train(agent, collect_key, COLLECT_SETTINGS, COLLECT_REWARD, forbid_bomb=True, **COLLECT)
eqx.tree_serialise_leaves(OUT / "collect.eqx", agent.network)
print(f"collect: {COLLECT['iterations']} iterations in {(time.time() - start) / 60:.1f} min, "
      f"coins at the end {collect_history.coins.tail(50).mean():.2f} of 8")

## Stage 2, once per entropy coefficient

Every run starts from the same weights and uses the same key, so the runs differ only in $c_H$.

In [ ]:
frames = [collect_history.assign(stage="collect", entropy_coef=np.nan)]
for c_H in ENTROPY_COEFS:
    start = time.time()
    _, history = train(agent, sweep_key, CRATE_SETTINGS, CRATE_REWARD, entropy=(c_H, c_H),
                       iterations=SWEEP["iterations"], learning_rate=SWEEP["learning_rate"])
    frames.append(history.assign(stage="crates", entropy_coef=c_H))
    tail = history.tail(50)
    print(f"c_H={c_H:.1e}: coins {tail.coins.mean():.2f}  bombs {tail.bombs.mean():.2f}  "
          f"death {tail.death.mean():.2f}  ({time.time() - start:.0f}s)")

history = pd.concat(frames, ignore_index=True)
history.to_csv(OUT / "history.csv", index=False)
(OUT / "config.json").write_text(json.dumps(dict(
    size=SIZE, seed=SEED, network="KartalCNNActorCriticNetwork", num_envs=NUM_ENVS, collect=COLLECT,
    sweep=dict(SWEEP, entropy_coefs=[float(c) for c in ENTROPY_COEFS]),
    collect_scenario=dict(coins=8, max_steps=96), crate_scenario=dict(coins=6, crate_density=0.18, max_steps=140),
    crate_reward="CoinPickupReward + UrgentCrateReward(0.3) + DeathPenalty(2.0) + UrgentStepPenalty",
), indent=2))
history.query("stage == 'crates'").groupby("entropy_coef")[["coins", "bombs", "death"]].apply(
    lambda d: d.tail(50).mean()).round(3)